In [ ]:
import importlib
import sys


def reimport_modules():
    """Force-reload reid and trackers after a fresh pip install (Colab-friendly)."""
    mods = [k for k in sys.modules if k.startswith(("reid", "trackers"))]
    for name in sorted(mods, reverse=True):
        importlib.reload(sys.modules[name])
    print(f"Reloaded {len(mods)} module(s).")


# Fine-tune ReID encoders on MOT17 (OSNet + FastReID)

Fine-tune **two** curated encoders on MOT17 train-half GT crops and compare them
against their published baselines on retrieval and BoT-SORT tracking:

| Key | Baseline (warm-start) | After fine-tune |
|-----|----------------------|-----------------|
| `osnet` | OSNet x1.0 / MSMT17 | OSNet on MOT17 train-half |
| `fastreid` | FastReID SBS ResNeSt50 / BoT-SORT MOT17 | Same stack, continued FT |

**Pipeline**

1. Download MOT17 train/val (Trackers CLI) + YOLOX val detections (gdown zip).
2. Crop pedestrians from Trackers' already-split `mot17/train` (full contents of that package); fine-tune **each** encoder on those identities.
3. Benchmark **retrieval** on GT crops from Trackers' `mot17/val` (mAP / Rank-1) — same sequence names, **later frames** (temporal hold-out; IDs can overlap).
4. Benchmark **tracking** on the same val-half split with BoT-SORT + ReID (HOTA / IDF1).

**Split safety:** Trackers' `mot17/train` / `mot17/val` are already the temporal halves. Crop with `split="full"` so we do not half them again. Tracking eval uses val-half frames + YOLOX detections (BoT-SORT / ByteTrack protocol).

Toggle which encoders to train/eval with `TRAIN_ENCODERS` in §2.

> **Runtime:** `Runtime → Change runtime type → T4 GPU` before running.
> FastReID is heavier (384×128 ResNeSt50); the notebook uses a smaller PK batch for it.


## 1. Install

Install **reid** and **trackers** from PyPI, plus notebook deps. **Run this cell before section 2 Setup.**

`trackers` is used for MOT17 download and the BoT-SORT A/B in later sections. Pass a loaded `reid.ReIDModel` into BoT-SORT when appearance ReID is enabled.


In [ ]:
!pip install -q --upgrade pip
# Colab ships CUDA torch; pip reuses it when it satisfies the requirement.
!pip install -q reid "trackers[reid]" matplotlib scikit-learn gdown

# Optional pre-release (no token) while PyPI builds catch up:
# !pip install -q "reid @ git+https://github.com/roboflow/re-ID.git@feat/training-port"
# !pip install -q "trackers @ git+https://github.com/roboflow/trackers.git@feat/core/reid-consume-reid-package"

reimport_modules()


In [ ]:
import warnings

import torch

warnings.filterwarnings("ignore")

device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}  |  Device: {device}")


## 2. Setup

Import the ReID training API, define paths / encoder recipes, and add **notebook-local**
helpers for YOLOX detections and embedding plots.

Only `load_mot_file` is imported from `trackers` — YOLOX I/O is defined below.
On Colab, data and outputs live under `/content/`.


In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import supervision as sv
from sklearn.decomposition import PCA
from trackers import load_mot_file
from trackers.eval import box_iou

from reid import ReIDEvaluator, ReIDModel
from reid.catalog import DEFAULT_MODEL, FASTREID_MOT17_SBS50
from reid.data import build_identity_crop_retrieval_split, load_identity_crops
from reid.data.mot import generate_mot_patches
from reid.training import TrainConfig, train_reid

try:
    import google.colab  # noqa: F401

    REPO_ROOT = Path("/content")
except ImportError:
    REPO_ROOT = Path("..").resolve()

OUT = REPO_ROOT / "outputs"
MOT17_TRAIN = REPO_ROOT / "mot17" / "train"
MOT17_VAL = REPO_ROOT / "mot17" / "val"
YOLOX_ROOT = REPO_ROOT / "MOT17_yolox_dets"
YOLOX_VAL_DIR = YOLOX_ROOT / "val"
YOLOX_FILE_ID = "1BuXtPWf8QbPU_y1i2xY2IbTE-rj3l6qT"
PATCH_DIR = OUT / "reid_mot17_crops_train"
PATCH_VAL_DIR = OUT / "reid_mot17_crops_val"
TRACK_OUT = OUT / "trackers_reid_outputs"

VAL_SEQS = [
    "MOT17-02-FRCNN",
    "MOT17-04-FRCNN",
    "MOT17-05-FRCNN",
    "MOT17-09-FRCNN",
    "MOT17-10-FRCNN",
    "MOT17-11-FRCNN",
    "MOT17-13-FRCNN",
]

# Which encoders to fine-tune and include in §7–§9. Use a subset for a quicker run.
TRAIN_ENCODERS = ["osnet", "fastreid"]

# Shared training defaults; per-encoder overrides in ENCODERS below.
_COMMON = dict(
    weight_decay=5e-4,
    margin=0.3,
    label_smoothing=0.1,
    triplet_weight=1.0,
    use_center=False,
    amp=True,
    device="auto",
    num_workers=4,
    log_interval=20,
)

# Two recipes in one notebook:
# - OSNet: MSMT17 → MOT17 domain adaptation (256×128).
# - FastReID: BoT-SORT MOT17 SBS checkpoint continued FT (384×128; smaller PK batch).
ENCODERS = {
    "osnet": {
        "baseline_label": "OSNet (MSMT17)",
        "finetuned_label": "OSNet (MOT17 finetuned)",
        "pretrained": DEFAULT_MODEL,
        "checkpoint_dir": OUT / "reid_mot17_osnet",
        # BoT-SORT default θ_emb is 0.25; OSNet distances often need a higher gate — §8c retunes.
        "default_appearance_threshold": 0.25,
        "config": TrainConfig(
            epochs=40,
            p=8,
            k=4,
            lr=3e-4,
            warmup_epochs=5,
            lr_milestones=(30, 35),
            freeze_backbone_epochs=3,
            **_COMMON,
        ),
    },
    "fastreid": {
        "baseline_label": "FastReID SBS (BoT-SORT MOT17)",
        "finetuned_label": "FastReID SBS (MOT17 finetuned)",
        "pretrained": FASTREID_MOT17_SBS50,
        "checkpoint_dir": OUT / "reid_mot17_fastreid",
        "default_appearance_threshold": 0.25,
        "config": TrainConfig(
            # Official weights are already MOT17 train-half; continued FT with a lower LR.
            epochs=20,
            p=4,
            k=4,
            lr=1e-4,
            warmup_epochs=2,
            lr_milestones=(12, 16),
            freeze_backbone_epochs=5,
            **_COMMON,
        ),
    },
}

assert all(k in ENCODERS for k in TRAIN_ENCODERS), TRAIN_ENCODERS


def yolox_det_path(seq: str) -> Path:
    return YOLOX_VAL_DIR / f"{seq.replace('-FRCNN', '')}_val.txt"


def yolox_det_frame_offset(det_path: Path) -> int:
    """YOLOX val dets use absolute frame ids (302+); val-half GT/images use 1..N."""
    min_frame = None
    with det_path.open() as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) < 6:
                continue
            frame = int(float(parts[0]))
            min_frame = frame if min_frame is None else min(min_frame, frame)
    return (min_frame - 1) if min_frame and min_frame > 1 else 0


def load_yolox_val_dets(det_path: Path, frame_offset: int = 0) -> dict[int, sv.Detections]:
    """Load YOLOX export: frame,x1,y1,x2,y2,score → {frame: Detections}."""
    frame_map: dict[int, list[list[float]]] = {}
    with det_path.open() as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) < 6:
                continue
            frame = int(float(parts[0])) - frame_offset
            if frame < 1:
                continue
            x1, y1, x2, y2, score = map(float, parts[1:6])
            if score <= 0:
                continue
            frame_map.setdefault(frame, []).append([x1, y1, x2, y2, score])

    return {
        frame: sv.Detections(
            xyxy=np.array(boxes, dtype=np.float32)[:, :4],
            confidence=np.array(boxes, dtype=np.float32)[:, 4],
        )
        for frame, boxes in frame_map.items()
    }


def load_val_dets(det_path: Path) -> dict[int, sv.Detections]:
    """YOLOX dets with frame ids remapped to match val-half images."""
    return load_yolox_val_dets(det_path, frame_offset=yolox_det_frame_offset(det_path))


def match_det_boxes_to_gt_ids(gt_frame, det_xyxy: np.ndarray, min_iou: float = 0.5) -> np.ndarray:
    gt_xyxy = sv.xywh_to_xyxy(gt_frame.boxes)
    keep = (gt_frame.confidences > 0) & (gt_frame.classes == 1)
    gt_xyxy, gt_ids = gt_xyxy[keep], gt_frame.ids[keep]
    if len(gt_xyxy) == 0:
        return np.full(len(det_xyxy), -1, dtype=np.int64)
    ious = box_iou(det_xyxy.astype(np.float64), gt_xyxy.astype(np.float64))
    assigned = np.full(len(det_xyxy), -1, dtype=np.int64)
    for i in range(len(det_xyxy)):
        j = int(np.argmax(ious[i]))
        if ious[i, j] >= min_iou:
            assigned[i] = int(gt_ids[j])
    return assigned


def collect_yolox_gt_embeddings(
    model: ReIDModel,
    sequences: list[str],
    *,
    frame_stride: int = 5,
    max_frames_per_seq: int | None = None,
    min_det_conf: float = 0.5,
) -> tuple[np.ndarray, np.ndarray]:
    """Embed YOLOX detections matched to GT track ids (tracking-domain crops)."""
    embeddings, gt_ids = [], []
    for seq in sequences:
        gt_by_frame = load_mot_file(MOT17_VAL / seq / "gt" / "gt.txt")
        det_by_frame = load_val_dets(yolox_det_path(seq))
        frame_paths = sorted((MOT17_VAL / seq / "img1").glob("*.jpg"))
        frame_indices = list(range(1, len(frame_paths) + 1, frame_stride))
        if max_frames_per_seq is not None:
            frame_indices = frame_indices[:max_frames_per_seq]

        for frame_idx in frame_indices:
            dets = det_by_frame.get(frame_idx)
            gt_frame = gt_by_frame.get(frame_idx)
            if dets is None or len(dets) == 0 or gt_frame is None:
                continue
            dets = dets[dets.confidence >= min_det_conf]
            if len(dets) == 0:
                continue

            bgr = cv2.imread(str(frame_paths[frame_idx - 1]))
            matched = match_det_boxes_to_gt_ids(gt_frame, dets.xyxy)
            feats = model.extract_features(dets, bgr)
            for i in range(len(dets)):
                if matched[i] < 0:
                    continue
                embeddings.append(feats[i])
                gt_ids.append(int(matched[i]))

    if not embeddings:
        raise RuntimeError("No GT-matched YOLOX embeddings collected — run §3 downloads first.")
    return np.stack(embeddings), np.array(gt_ids, dtype=np.int64)


def sample_intra_inter_cosine_distances(
    embeddings: np.ndarray,
    gt_ids: np.ndarray,
    *,
    n_intra: int = 8000,
    n_inter: int = 8000,
    seed: int = 0,
) -> tuple[np.ndarray, np.ndarray]:
    """Sample cosine distances for same-GT-id pairs vs different-GT-id pairs."""
    normed = embeddings / (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-12)
    rng = np.random.default_rng(seed)
    by_id: dict[int, list[int]] = {}
    for idx, pid in enumerate(gt_ids):
        by_id.setdefault(int(pid), []).append(idx)

    ids_with_pair = [pid for pid, idxs in by_id.items() if len(idxs) >= 2]
    all_ids = list(by_id.keys())
    if not ids_with_pair:
        raise ValueError("Need at least one GT id with two embeddings for intra-class distances.")

    intra = np.empty(n_intra)
    for k in range(n_intra):
        pid = rng.choice(ids_with_pair)
        i, j = rng.choice(by_id[pid], size=2, replace=False)
        intra[k] = 1.0 - float(normed[i] @ normed[j])

    inter = np.empty(n_inter)
    for k in range(n_inter):
        pid_a, pid_b = rng.choice(all_ids, size=2, replace=False)
        i = rng.choice(by_id[pid_a])
        j = rng.choice(by_id[pid_b])
        inter[k] = 1.0 - float(normed[i] @ normed[j])

    return intra, inter


def suggest_appearance_threshold(intra: np.ndarray, inter: np.ndarray) -> float:
    """Heuristic θ_emb from BoT-SORT-style distance histograms (§8c → §9)."""
    lo = float(np.quantile(intra, 0.90))
    hi = float(np.quantile(inter, 0.10))
    return (lo + hi) / 2


def plot_intra_inter_distance_histogram(
    intra: np.ndarray,
    inter: np.ndarray,
    *,
    title: str,
    ax: plt.Axes | None = None,
    appearance_threshold: float | None = None,
) -> plt.Axes:
    """BoT-SORT-style histogram of embedding distances (same id vs different id)."""
    ax = ax or plt.gca()
    bins = np.linspace(0, 1.2, 50)
    ax.hist(intra, bins=bins, alpha=0.55, density=True, label="Same GT ID", color="#3366CC")
    ax.hist(inter, bins=bins, alpha=0.55, density=True, label="Different GT ID", color="#DC3912")
    if appearance_threshold is not None:
        ax.axvline(
            appearance_threshold,
            color="black",
            ls="--",
            lw=1.5,
            label=f"θ_emb={appearance_threshold:.2f}",
        )
    ax.set(xlabel="Cosine distance", ylabel="Density", title=title)
    ax.legend()
    ax.grid(True, alpha=0.25)
    return ax


def plot_pca_side_by_side(
    emb_left: np.ndarray,
    emb_right: np.ndarray,
    labels: np.ndarray,
    *,
    left_title: str,
    right_title: str,
    suptitle: str,
) -> None:
    """Side-by-side 2D PCA scatter; same color = same identity label."""
    coords_l = PCA(n_components=2, random_state=0).fit_transform(emb_left)
    coords_r = PCA(n_components=2, random_state=0).fit_transform(emb_right)
    unique = np.unique(labels)
    cmap = plt.colormaps["tab20"].resampled(max(len(unique), 1))
    colors = {pid: cmap(i % 20) for i, pid in enumerate(unique)}

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)
    for ax, coords, title in [(axes[0], coords_l, left_title), (axes[1], coords_r, right_title)]:
        for pid in unique:
            mask = labels == pid
            ax.scatter(coords[mask, 0], coords[mask, 1], s=26, alpha=0.85, color=colors[pid])
        ax.set(title=title, xlabel="PC1")
        ax.grid(True, alpha=0.25)
    axes[0].set_ylabel("PC2")
    fig.suptitle(suptitle, y=1.02)
    plt.tight_layout()
    plt.show()


print("Setup complete.")
print("TRAIN_ENCODERS:", TRAIN_ENCODERS)
for key in TRAIN_ENCODERS:
    recipe = ENCODERS[key]
    cfg = recipe["config"]
    print(
        f"  {key}: pretrained={recipe['pretrained']!r}  "
        f"epochs={cfg.epochs} p={cfg.p} k={cfg.k} lr={cfg.lr}  "
        f"→ {recipe['checkpoint_dir'].name}"
    )


---
## 3. Download data

### 3a. MOT17 frames and ground truth

Downloads train + val into `mot17/`. Train sequences supply GT crops; val sequences
supply GT + images for the tracking benchmark.


In [ ]:
import subprocess

FORCE_DOWNLOAD_MOT17 = False

train_seqs = sum(1 for _ in MOT17_TRAIN.glob("MOT17-*/gt/gt.txt"))
mot17_ready = train_seqs >= len(VAL_SEQS) and all((MOT17_VAL / seq / "gt" / "gt.txt").is_file() for seq in VAL_SEQS)

if FORCE_DOWNLOAD_MOT17 or not mot17_ready:
    subprocess.run(  # noqa: S603
        [
            sys.executable,
            "-m",
            "trackers.scripts",
            "download",
            "mot17",
            "--split",
            "train,val",
            "--asset",
            "annotations,frames",
            "-o",
            str(REPO_ROOT),
        ],
        check=True,
    )
else:
    print("MOT17 already present.")

train_seqs = sum(1 for _ in MOT17_TRAIN.glob("MOT17-*/gt/gt.txt"))
print(f"Train sequences ready: {train_seqs}/{len(VAL_SEQS)}")


### 3b. YOLOX val detections

Public YOLOX detector outputs for the MOT17 val-half split (same zip used in
BoT-SORT / ByteTrack eval notebooks). Extracted to `MOT17_yolox_dets/val/`.


In [ ]:
import zipfile

import gdown

FORCE_DOWNLOAD_YOLOX = False
zip_path = YOLOX_ROOT / "yolox_detections_MOT17.zip"

yolox_ready = YOLOX_VAL_DIR.is_dir() and len(list(YOLOX_VAL_DIR.glob("MOT17-*_val.txt"))) >= len(VAL_SEQS)

if FORCE_DOWNLOAD_YOLOX or not yolox_ready:
    YOLOX_ROOT.mkdir(parents=True, exist_ok=True)
    gdown.download(id=YOLOX_FILE_ID, output=str(zip_path), quiet=False)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(YOLOX_ROOT)
else:
    print("YOLOX detections already present.")

n_dets = len(list(YOLOX_VAL_DIR.glob("MOT17-*_val.txt")))
print(f"YOLOX val det files: {n_dets}/{len(VAL_SEQS)}")


---
## 4. Generate GT crops

Two crop roots from Trackers' already-split MOT17 packages (`split="full"` — do not half again):

| Directory | Source | Frames | Used in |
|-----------|--------|--------|---------|
| `reid_mot17_crops_train` | `mot17/train` | full contents of train package (temporal train-half) | §6 fine-tune (all identities) |
| `reid_mot17_crops_val` | `mot17/val` | full contents of val package (temporal val-half) | §7 retrieval (all val identities) |

Train and val use the **same sequence names** with **temporally held-out frames**. Pedestrian IDs often overlap across the cut; this is not an open-set identity split. Val-half frames align with §9 tracking.
Both OSNet and FastReID train on the **same** crop folders (each recipe applies its own input size / augmentations).


In [ ]:
# Trackers' mot17/train and mot17/val are already temporal halves — use full.
train_stats = generate_mot_patches(
    MOT17_TRAIN,
    PATCH_DIR,
    split="full",
    min_visibility=0.3,
    min_side=16,
)
val_stats = generate_mot_patches(
    MOT17_VAL,
    PATCH_VAL_DIR,
    split="full",
    min_visibility=0.3,
    min_side=16,
)

print(f"Train crops: {train_stats.num_identities} identities, {train_stats.num_crops} crops")
print(f"Val crops:   {val_stats.num_identities} identities, {val_stats.num_crops} crops")


## 5. Retrieval eval split (val frames)

Fine-tune on **every** identity in the train-half crops (§6). Build query/gallery from
**val-half** GT crops of the same sequences — temporally held-out frames (IDs may
overlap with training). Matches the same val domain as §9 tracking.

For an open-set identity hold-out instead, use `reid.data.build_identity_crop_holdout_split`.


In [ ]:
val_split = load_identity_crops(PATCH_VAL_DIR)
query, gallery = build_identity_crop_retrieval_split(val_split, queries_per_id=1)
num_val_retrieval_ids = len(set(query.pids.tolist()))

print(f"Train identities (fine-tune): all {train_stats.num_identities} under {PATCH_DIR.name}")
print(f"Val retrieval identities:     {num_val_retrieval_ids}  (query={len(query)}, gallery={len(gallery)})")


---
## 6. Fine-tune encoders

For each key in `TRAIN_ENCODERS`:

- **OSNet** warm-starts from the curated MSMT17 alias (out-of-domain → MOT17).
- **FastReID** warm-starts from BoT-SORT's MOT17 SBS ResNeSt50 checkpoint (already
  MOT17 train-half; this is continued fine-tuning with a fresh classifier head).

Loss = label-smoothing CE + batch-hard triplet. Classifier head is always re-initialized
for the MOT17 identity count. Shared crop root; each recipe resolves its own
preprocessing via `resolve_load_plan` (256×128 OSNet, 384×128 FastReID).


In [ ]:
train_results = {}

for key in TRAIN_ENCODERS:
    recipe = ENCODERS[key]
    print(f"\n=== Fine-tuning {key}: {recipe['baseline_label']} → {recipe['finetuned_label']} ===")
    result = train_reid(
        PATCH_DIR,
        recipe["config"],
        pretrained=recipe["pretrained"],
        output_dir=recipe["checkpoint_dir"],
    )
    train_results[key] = result
    print(
        f"Checkpoint: {result.output_dir}\n"
        f"Classes: {result.num_classes}  |  Final loss: {result.history[-1]['total']:.4f}"
    )


In [ ]:
n = len(train_results)
fig, axes = plt.subplots(n, 3, figsize=(13, 3.2 * n), squeeze=False)

for row, key in enumerate(train_results):
    history = train_results[key].history
    epochs = [r["epoch"] for r in history]
    for col, (metric, title) in enumerate(
        (("ce", "Cross-entropy"), ("triplet", "Triplet"), ("total", "Total"))
    ):
        ax = axes[row, col]
        ax.plot(epochs, [r[metric] for r in history], marker="o", ms=3)
        ax.set(xlabel="epoch", ylabel=metric, title=f"{key}: {title}")
        ax.grid(True, alpha=0.25)

fig.suptitle("Fine-tune loss curves", y=1.01)
plt.tight_layout()
plt.show()


---
## 7. Retrieval benchmark

For each encoder family, score the published **baseline** and the **MOT17 finetuned**
checkpoint on the §5 val-sequence query/gallery split.


In [ ]:
models = {}

for key in TRAIN_ENCODERS:
    recipe = ENCODERS[key]
    models[f"{key}_baseline"] = ReIDModel.from_pretrained(recipe["pretrained"])
    models[f"{key}_finetuned"] = ReIDModel.from_pretrained(str(recipe["checkpoint_dir"]))

print(f"{'model':<40}  mAP     Rank-1")
print("-" * 58)
for key in TRAIN_ENCODERS:
    recipe = ENCODERS[key]
    for suffix, label_key in (("baseline", "baseline_label"), ("finetuned", "finetuned_label")):
        model_key = f"{key}_{suffix}"
        m = ReIDEvaluator(models[model_key]).evaluate(query, gallery, verbose=False).metrics
        print(
            f"{recipe[label_key]:<40}  "
            f"{m.mean_average_precision:5.1f}%  {m.rank1:5.1f}%"
        )


---
## 8. Visualize embeddings

### 8a. Val-sequence GT crops

2D PCA on embeddings from **val-sequence GT crops** (same domain as §7). One row per
encoder family: baseline vs finetuned. Same color = same identity.


In [ ]:
MAX_VIZ = 400
paths = list(query.image_paths) + list(gallery.image_paths)
pids = np.concatenate([query.pids, gallery.pids])

if len(paths) > MAX_VIZ:
    keep = np.sort(np.random.default_rng(0).choice(len(paths), MAX_VIZ, replace=False))
    paths, pids = [paths[i] for i in keep], pids[keep]

for key in TRAIN_ENCODERS:
    recipe = ENCODERS[key]
    emb_base = models[f"{key}_baseline"].extract_features_from_paths(paths, batch_size=64)
    emb_ft = models[f"{key}_finetuned"].extract_features_from_paths(paths, batch_size=64)
    plot_pca_side_by_side(
        emb_base,
        emb_ft,
        pids,
        left_title=recipe["baseline_label"],
        right_title=recipe["finetuned_label"],
        suptitle=f"{key}: val-sequence GT crops — {len(paths)} points, {len(np.unique(pids))} identities",
    )


### 8b. YOLOX detector crops

Same PCA comparison on **YOLOX detections** matched to val-half GT by IoU — the crop
domain BoT-SORT actually sees at inference.


In [ ]:
SEQ = "MOT17-02-FRCNN"

for key in TRAIN_ENCODERS:
    recipe = ENCODERS[key]
    emb_base, gt_ids = collect_yolox_gt_embeddings(models[f"{key}_baseline"], [SEQ], frame_stride=5)
    emb_ft, _ = collect_yolox_gt_embeddings(models[f"{key}_finetuned"], [SEQ], frame_stride=5)

    if len(gt_ids) > MAX_VIZ:
        keep = np.sort(np.random.default_rng(0).choice(len(gt_ids), MAX_VIZ, replace=False))
        emb_base, emb_ft, gt_ids = emb_base[keep], emb_ft[keep], gt_ids[keep]

    plot_pca_side_by_side(
        emb_base,
        emb_ft,
        gt_ids,
        left_title=recipe["baseline_label"],
        right_title=recipe["finetuned_label"],
        suptitle=f"{key}: YOLOX crops — {SEQ} ({len(gt_ids)} points, {len(np.unique(gt_ids))} identities)",
    )


### 8c. Same-ID vs different-ID distance distributions

Classic ReID diagnostic (see BoT-SORT paper): histogram of **cosine distance** between
detector-crop embeddings from the **same GT track** vs **different GT tracks**.

The dashed **θ_emb** line is the appearance gate used in §9 (`d_app < θ_emb`).
BoT-SORT's paper default is `0.25`; §2 seeds each encoder with that, then this cell
replaces it with a histogram heuristic. Override in the next cell after inspecting overlap.


In [ ]:
appearance_threshold = {
    key: ENCODERS[key]["default_appearance_threshold"] for key in TRAIN_ENCODERS
}
distance_stats = {}

for key in TRAIN_ENCODERS:
    recipe = ENCODERS[key]
    emb_base, gt_ids = collect_yolox_gt_embeddings(
        models[f"{key}_baseline"],
        VAL_SEQS,
        frame_stride=5,
        max_frames_per_seq=30,
    )
    emb_ft, _ = collect_yolox_gt_embeddings(
        models[f"{key}_finetuned"],
        VAL_SEQS,
        frame_stride=5,
        max_frames_per_seq=30,
    )

    intra_base, inter_base = sample_intra_inter_cosine_distances(emb_base, gt_ids, seed=0)
    intra_ft, inter_ft = sample_intra_inter_cosine_distances(emb_ft, gt_ids, seed=0)
    distance_stats[key] = {
        "baseline": (intra_base, inter_base),
        "finetuned": (intra_ft, inter_ft),
    }

    # Use the finetuned histogram for the tracking gate; fall back to baseline if needed.
    appearance_threshold[key] = suggest_appearance_threshold(intra_ft, inter_ft)
    print(f"{key}: θ_emb={appearance_threshold[key]:.3f} (BoT-SORT default 0.25)")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True, sharey=True)
    plot_intra_inter_distance_histogram(
        intra_base,
        inter_base,
        title=recipe["baseline_label"],
        ax=axes[0],
        appearance_threshold=appearance_threshold[key],
    )
    plot_intra_inter_distance_histogram(
        intra_ft,
        inter_ft,
        title=recipe["finetuned_label"],
        ax=axes[1],
        appearance_threshold=appearance_threshold[key],
    )
    fig.suptitle(
        f"{key}: YOLOX crop cosine distances — {len(gt_ids)} embeddings, "
        f"{len(np.unique(gt_ids))} GT ids",
        y=1.02,
    )
    plt.tight_layout()
    plt.show()

    for suffix, (intra, inter) in distance_stats[key].items():
        theta = appearance_threshold[key]
        print(
            f"  {suffix}: same-id mean={intra.mean():.3f}  diff-id mean={inter.mean():.3f}  "
            f"gap={inter.mean() - intra.mean():.3f}  θ_emb={theta:.3f} "
            f"(cos sim > {1 - theta:.2f})"
        )


In [ ]:
# Optional manual overrides after inspecting the histograms (cosine distance).
# appearance_threshold["osnet"] = 0.40
# appearance_threshold["fastreid"] = 0.25


---
## 9. Tracking benchmark

Run **BoT-SORT + CMC** on all seven val sequences. GT and frames come from `mot17/val/`;
detections come from the YOLOX zip (§3b).

Configs:

- BoT-SORT without ReID
- For each encoder in `TRAIN_ENCODERS`: baseline ReID and MOT17-finetuned ReID


In [ ]:
from trackers import BoTSORTTracker
from trackers.eval import evaluate_mot_sequences
from trackers.eval.results import BenchmarkResult
from trackers.io.mot import _MOTOutput

# Set False to reuse cached preds under outputs/trackers_reid_outputs/
RERUN = {"botsort_baseline": False}
for key in TRAIN_ENCODERS:
    RERUN[f"botsort_reid_{key}_baseline"] = True
    RERUN[f"botsort_reid_{key}_finetuned"] = True

TRACK_OUT.mkdir(parents=True, exist_ok=True)
seqmap_path = TRACK_OUT / "MOT17-val.txt"
seqmap_path.write_text("name\n" + "\n".join(VAL_SEQS) + "\n")


In [ ]:
def track_val_split(name: str, tracker_factory) -> BenchmarkResult:
    pred_dir = TRACK_OUT / name / "preds"
    cache = TRACK_OUT / name / "eval_results.json"

    if not RERUN.get(name, True) and cache.is_file():
        print("  (cached)")
        return BenchmarkResult.load(cache)

    pred_dir.mkdir(parents=True, exist_ok=True)
    for seq in VAL_SEQS:
        det_by_frame = load_val_dets(yolox_det_path(seq))
        frame_paths = sorted((MOT17_VAL / seq / "img1").glob("*.jpg"))
        tracker = tracker_factory()

        with _MOTOutput(pred_dir / f"{seq}.txt") as writer:
            for frame_idx, frame_path in enumerate(frame_paths, start=1):
                dets = det_by_frame.get(frame_idx, sv.Detections.empty())
                tracked = tracker.update(dets, cv2.imread(str(frame_path)))
                if tracked.tracker_id is not None:
                    tracked = tracked[tracked.tracker_id != -1]
                writer.write(frame_idx, tracked)

    result = evaluate_mot_sequences(
        gt_dir=MOT17_VAL,
        tracker_dir=pred_dir,
        seqmap=seqmap_path,
        metrics=["CLEAR", "HOTA", "Identity"],
    )
    result.save(cache)
    return result


experiments = [
    ("BoT-SORT (no ReID)", "botsort_baseline", lambda: BoTSORTTracker(enable_cmc=True, reid_model=None)),
]
for key in TRAIN_ENCODERS:
    recipe = ENCODERS[key]
    theta = appearance_threshold[key]
    experiments.append(
        (
            f"BoT-SORT + {recipe['baseline_label']}",
            f"botsort_reid_{key}_baseline",
            lambda m=models[f"{key}_baseline"], t=theta: BoTSORTTracker(
                enable_cmc=True,
                reid_model=m,
                reid_ema_alpha=0.9,
                appearance_threshold=t,
            ),
        )
    )
    experiments.append(
        (
            f"BoT-SORT + {recipe['finetuned_label']}",
            f"botsort_reid_{key}_finetuned",
            lambda m=models[f"{key}_finetuned"], t=theta: BoTSORTTracker(
                enable_cmc=True,
                reid_model=m,
                reid_ema_alpha=0.9,
                appearance_threshold=t,
            ),
        )
    )

results = {}
for label, name, factory in experiments:
    print(label)
    result = track_val_split(name, factory)
    results[name] = result
    agg = result.aggregate
    print(
        f"  HOTA={agg.HOTA.HOTA * 100:6.2f}  "
        f"MOTA={agg.CLEAR.MOTA * 100:6.2f}  "
        f"IDF1={agg.Identity.IDF1 * 100:6.2f}  "
        f"IDSW={agg.CLEAR.IDSW}"
    )

print("\nFinetuned − baseline (ReID configs):")
for key in TRAIN_ENCODERS:
    base = results[f"botsort_reid_{key}_baseline"].aggregate
    ft = results[f"botsort_reid_{key}_finetuned"].aggregate
    print(
        f"  {key}:  "
        f"delta HOTA {100 * (ft.HOTA.HOTA - base.HOTA.HOTA):+6.2f}  "
        f"delta IDF1 {100 * (ft.Identity.IDF1 - base.Identity.IDF1):+6.2f}"
    )
